# OverSampling
______

data_science -> balanced -> OverSampling -> Exemplos

## Configurações necessárias para rodar as funções

In [0]:
%run
Utils/configuration

imblearn is installed

Out[106]: DataFrame[]

## Pacotes necessarios para os exemplos

In [0]:
from sklearn.datasets import load_iris

## Carregar os dados e transformar

In [0]:
iris = load_iris()
df = pd.DataFrame(data=iris.data, 
                  columns=iris.feature_names)
df['species'] = iris.target
df['species'] = df['species'].map({0: 'Iris-virginica', 1:'Iris-setosa', 2:'Iris-versicolor'})
df = spark.createDataFrame(df)

In [0]:
df = df.withColumnRenamed('sepal length (cm)', "sepal_length")
df = df.withColumnRenamed('sepal width (cm)', "sepal_Width")
df = df.withColumnRenamed('petal length (cm)', "petal_length")
df = df.withColumnRenamed('petal width (cm)', "petal_width")

### Desbalancear a massa de dados para utilizar as funcionalidades.

In [0]:
df = df.filter(df.sepal_length != 6.2).drop()

In [0]:
df.groupBy('species').count().show()

+---------------+-----+
 species|count|
+---------------+-----+
 Iris-virginica| 50|
 Iris-setosa| 48|
Iris-versicolor| 48|
+---------------+-----+

## Methods

### balanced_Random_Over_Sampler
#### Naive random over-sampling
* gera novas amostras de forma aleatória a fim de equiparar as classes do dataset.
* permite amostrar dados heterogeneos (que possuem strings)
* utiliza a  formula de smooth bootstrap para suavizar os espaços entre as amostras 

---

#### Documentação 
___

* https://adb-6226397775262915.15.azuredatabricks.net/?o=6226397775262915#notebook/97507982832939/command/97507982832942

#### Exemplo

In [0]:
df_balanced = balanced_Random_Over_Sampler(df, "species")

Retirada de registro entre as classes usando o parametro sampling_strategy: auto
Controlador de randomização do algoritmo: None
Controlador de de contração da matriz de covariância: None
Resampled dataset shape Counter({2: 50, 0: 50, 1: 50})

### balanced_SMOTE
#### SMOTE
* É uma das técnicas mais famosas quando falamos sobre over sampling.
* Equilibra a distribuição de classes aumentando aleatoriamente os exemplos das classes minoritátias, replicando-as
* Esses registros são gerados por interpolação linear
* Os registros selecionam aleatoriamente um ou mais dos K-vizinhos para cada exemplo da classe minoritária
* Diferente da ADASYN o Smote não fará distinção entre as amostras usando o a regra NN
* Conecta inliers e outliers

#### Documentação
___

* https://adb-6226397775262915.15.azuredatabricks.net/?o=6226397775262915#notebook/97507982832939/command/97507982832944

#### Exemplo

In [0]:
df_balanced = balanced_SMOTE(df, "species")

Retirada de registro entre as classes usando o parametro sampling_strategy: auto
Controlador de randomização do algoritmo: None
Número de K para construir as amostras sintéticas: 5
Numero de CPUs usados durante a cross validation: None
Resampled dataset shape Counter({2: 50, 0: 50, 1: 50})

### balanced_SMOTENC
#### SMOTENC 
* Lida com variáveis categóricas
* usado para variáveis numéricas e categóricas
* é necessário fornecer essa informação no SMOTEC por meio do parâmetro categorical_features passando os indices desses recursos ou uma máscara BOOLEANA marcando esses recursos

#### Documentação
___

* https://adb-6226397775262915.15.azuredatabricks.net/?o=6226397775262915#notebook/97507982832939/command/97507982832947

#### Exemplo

In [0]:
df_balanced = balanced_SMOTENC(df, "species", categorical_features = [0,2])

Retirada de registro entre as classes usando o parametro sampling_strategy: auto
Controlador de randomização do algoritmo: None
Número de K para construir as amostras sintéticas: 5
Numero de CPUs usados durante a cross validation: None
Resampled dataset shape Counter({2: 50, 0: 50, 1: 50})

### balanced_SMOTEN
####SMOTEN
* dataset de dados categóricos
* o algoritmo pode mudar do SMOTENC em duas formas
1) o algotitmo KNN não confia na distancia euclidiana, assim, a VDM (métrica do valor da diferença) também pode ser implementado pelo paramentro ValueDifferencMetric
2) uma nova amostra é gerada onde cada valor da característica corresponde à categoria mais comum vista nas amostras vizinhas pertencentes à mesma classe. 
PS: espera-se que o modelo gere uma nova categoria junto com a classe minoritária

#### Documentação
___

* https://adb-6226397775262915.15.azuredatabricks.net/?o=6226397775262915#notebook/97507982832939/command/97507982832958

#### Exemplo

In [0]:
df_balanced = balanced_SMOTEN(df, "species")

Retirada de registro entre as classes usando o parametro sampling_strategy: auto
Controlador de randomização do algoritmo: None
Número de K para construir as amostras sintéticas: 5
Numero de CPUs usados durante a cross validation: None
Resampled dataset shape Counter({2: 50, 0: 50, 1: 50})

### balanced_ADASYN
#### ADASYN
* Baseado na premissa do SMOTE. Usa a distribuição de densidade como um critério para decidir automaticamente o numero de dados sintéticos que precisam ser gerados para cada exemplo da classe minoritária.
* irá focar em gerar  amostras próximas as amostras originais que são classificadas erroneamente usando a classificação básica do KNN
* pode se concentrar apenas em outliers

#### Documentação
___

* https://adb-6226397775262915.15.azuredatabricks.net/?o=6226397775262915#notebook/97507982832939/command/97507982832960

#### Exemplo

In [0]:
df_balanced = balanced_ADASYN(df, np.array("species"))

--------------------------------------------------------------------------- 
 ValueError Traceback (most recent call last)
 <command-1513522336282130> in <module> 
 ----> 1 df_balanced = balanced_ADASYN ( df , "species" ) 

 <command-1513522336282051> in balanced_ADASYN (df, target, sampling_strategy, random_state, n_neighbors, n_jobs) 
 9 '''
 10 ada = ADASYN ( sampling_strategy = sampling_strategy , random_state = random_state , n_neighbors = n_neighbors , n_jobs = n_jobs ) 
 ---> 11 X_res , y_res = ada . fit_resample ( X , y ) 
 12 print ( f'Retirada de registro entre as classes usando o parametro sampling_strategy: {sampling_strategy}' ) 
 13 print ( f'Controlador de randomização do algoritmo: {random_state}' ) 

 /databricks/python/lib/python3.8/site-packages/imblearn/base.py in fit_resample (self, X, y) 
 81 )
 82 
 ---> 83 output = self . _fit_resample ( X , y ) 
 84 
 85 y_ = (

 /databricks/python/lib/python3.8/site-packages/imblearn/over_sampling/_adasyn.py in _fit_resample (self, X, y) 
 145 n_samples = np . sum ( n_samples_generate ) 
 146 if not n_samples : 
 --> 147 raise ValueError(
 148 "No samples will be generated with the provided ratio settings." 
 149 )

 ValueError : No samples will be generated with the provided ratio settings.

### balanced_borderline_SMOTE
#### BorderlineSMOTE
* identifica amostras proximas a superficie de decisão, e aplica o smothe somente em instancias do conjunto DANGER.
* esse conjunto será formado por instancias que possuem mais vizinhos da classe majoritária do que da classe minoritária

#### Documentação
___

* https://adb-6226397775262915.15.azuredatabricks.net/?o=6226397775262915#notebook/97507982832939/command/97507982832962

#### Exemplo

In [0]:
df_balanced = balanced_borderline_SMOTE(df, "species")

Retirada de registro entre as classes usando o parametro sampling_strategy: auto
Controlador de randomização do algoritmo: None
Número de K para construir as amostras sintéticas: 5
Numero de CPUs usados durante a cross validation: None
Determinador da classe menor se está na condição danger: 10
Tipo do algoritmo borderline: borderline-1
Resampled dataset shape Counter({2: 50, 0: 50, 1: 50})

### balanced_KMeansSMOTE
#### K-Means SMOTE
* o algoritmo utiliza 3 passos
1) clusteriza todo o input usando o algoritmo k-means
2) distribui o numero de amostras geradas por meio do cluster - selecionando clusters que tem um numero pequeno de classes e atribui mais amostras onde a distribuição é esparsa.
3) faz o oversampling com os clusters filtrados udando a Tecnica SMOTE.

#### Documentação
___

* https://adb-6226397775262915.15.azuredatabricks.net/?o=6226397775262915#notebook/97507982832939/command/97507982832965

#### Exemplo

In [0]:
df_balanced = balanced_KMeansSMOTE(df, "species")

Retirada de registro entre as classes usando o parametro sampling_strategy: auto
Controlador de randomização do algoritmo: None
Número de K para construir as amostras sintéticas: 2
Numero de CPUs usados durante a cross validation: None
Instancia K-means ou o numero de clusters a serem usados: None
Limite de clusters que serão sobreamostradas pelo SMOTE: auto
Densidade do cluster: auto
Resampled dataset shape Counter({0: 52, 1: 52, 2: 50})

### balanced_SVMSMOTE
#### SVMSMOTE
* variação do borderline SMOTE
* ao inves de usar o KNN para identificar a classificação incorreta como o borderline ela implementa o Suport Vector Machine
* A área limítrofe é aprocimada pelos vetores de suporte após o treinamento do classificador SVM no conjunto de dados original
* os dados sintéticos serão criados aleatoriamente ao longo das linhas que unem cada vetor de suporte da classe minoritária com vários de deus vizinhos mais próximos
* outra diferença do SVMSMOTE é que os dados são sintetizados longe da região de sobreposição de classes e se concentra mais em onde os dados são separados.

#### Documentação
___

* https://adb-6226397775262915.15.azuredatabricks.net/?o=6226397775262915#notebook/97507982832939/command/97507982832967

#### Exemplo

In [0]:
df_balanced = balanced_SVMSMOTE(df, "species")

Retirada de registro entre as classes usando o parametro sampling_strategy: auto
Controlador de randomização do algoritmo: None
Número de K para construir as amostras sintéticas: 5
Numero de CPUs usados durante a cross validation: None
Determinador da classe menor se está na condição danger: 10
Estimador SVM: None
Tamanho do extrapolador: 0.5
Resampled dataset shape Counter({2: 50, 0: 50, 1: 50})